# 👁️ Face Recognition — YOLO26m Training on LFW

**Project**: Face Recognition System — PFE Sonatel Academy  
**Author**: Ibrahima Gabar Diop

---

## Pipeline
1. Dataset exploration — CSV analysis, full LFW distribution
2. Balanced class selection — center-band strategy, `lfw_allnames.csv`
3. YOLO dataset build — train / val / test (70/15/15)
4. YOLO26m training — latest Ultralytics architecture (Jan. 2026)
5. Training curves analysis
6. Test set evaluation — mAP, precision, recall, confusion matrix
7. Inference visualization
8. Model export (.pt + .onnx)

---

### Architecture

| Model | Params | mAP50-95 COCO | T4 Latency |
|---|---|---|---|
| YOLO11m | 20.1M | 51.5 | 8.0ms |
| YOLO12m | 20.2M | 52.5 | 4.9ms |
| **YOLO26m** | — | **SOTA** | NMS-free |

**Class selection strategy**: filter `>= MIN_IMAGES`, sort ascending, pick center band → no Bush-type dominance, cap at `MAX_IMAGES` to equalize residual imbalance.

In [ ]:
# ── 0. GPU Compatibility — ensure PyTorch supports the assigned GPU ───────────
#
# Kaggle may assign a Tesla P100 (sm_60). PyTorch ≥ 2.3 dropped sm_60 support.
# We check via nvidia-smi (no torch import yet!) and install a compatible build
# BEFORE ultralytics/torch are first imported — no kernel restart needed.

import subprocess, sys

def _get_compute_cap():
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=15,
    )
    return r.stdout.strip() if r.returncode == 0 else None

cap = _get_compute_cap()
print(f"GPU compute capability : {cap or 'N/A (CPU mode)'}")

if cap:
    major = int(cap.split(".")[0])
    if major < 7:   # P100 = 6.0 → needs torch ≤ 2.2.x
        print(f"⚠️  sm_{cap.replace('.', '')} (P100) — PyTorch 2.10 requires sm_70+")
        print("   Installing PyTorch 2.2.2+cu118 which supports sm_60 …")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "torch==2.2.2+cu118",
            "torchvision==0.17.2+cu118",
            "--index-url", "https://download.pytorch.org/whl/cu118",
        ])
        print("✓ PyTorch 2.2.2+cu118 installed — sm_60 compatible")
    else:
        import torch
        print(f"✓ sm_{cap.replace('.', '')} — PyTorch {torch.__version__} OK")
else:
    print("ℹ️  No GPU — training will use CPU")

# Install remaining deps AFTER the torch swap so they see the right version
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics==8.4.37", "seaborn",
])
print("✓ ultralytics 8.4.37 + seaborn ready")

In [ ]:
# ── 2. Imports & Constants ────────────────────────────────────────────────────
import os, shutil, random, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from IPython.display import display, Image as IPImage
from ultralytics import YOLO

# ── Paths (exact structure confirmed by user)
DATASET_DIR   = Path("/kaggle/input/datasets/jessicali9530/lfw-dataset")
LFW_DIR       = DATASET_DIR / "lfw-deepfunneled" / "lfw-deepfunneled"
CSV_ALLNAMES  = DATASET_DIR / "lfw_allnames.csv"
CSV_DEV_TRAIN = DATASET_DIR / "peopleDevTrain.csv"
CSV_DEV_TEST  = DATASET_DIR / "peopleDevTest.csv"
CSV_PEOPLE    = DATASET_DIR / "people.csv"
CSV_PAIRS     = DATASET_DIR / "pairs.csv"

WORK_DIR  = Path("/kaggle/working")
DATA_DIR  = WORK_DIR / "dataset"
MODEL_DIR = WORK_DIR / "models"

# ── Dataset
N_CLASSES   = 20
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
SEED        = 42
MIN_IMAGES  = 30    # min images per person to be eligible
MAX_IMAGES  = 100   # cap per class to prevent dominance

# ── Model
BASE_MODEL = "yolo26m.pt"
RUN_NAME   = "face_yolo26m_v1"
EPOCHS     = 100
IMGSZ      = 224
BATCH      = 32

random.seed(SEED)
np.random.seed(SEED)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Sanity checks
for label, p in [
    ("DATASET_DIR",    DATASET_DIR),
    ("LFW_DIR",        LFW_DIR),
    ("lfw_allnames",   CSV_ALLNAMES),
    ("peopleDevTrain", CSV_DEV_TRAIN),
    ("peopleDevTest",  CSV_DEV_TEST),
]:
    print(f"  {'✓' if p.exists() else '✗ MISSING':<12} {label:<20} {p}")

print(f"\n  Base model : {BASE_MODEL}")
print(f"  GPU        : {os.popen('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null').read().strip() or 'Not detected'}")

## 1. Dataset Exploration

In [ ]:
# ── Load lfw_allnames.csv ─────────────────────────────────────────────────────
df_allnames = pd.read_csv(CSV_ALLNAMES)
df_allnames.columns = df_allnames.columns.str.strip()
print(f"lfw_allnames.csv — {df_allnames.shape[0]} rows  columns: {list(df_allnames.columns)}")
print(df_allnames.head(8).to_string(index=False))
print()

# Normalize column names
df_names = df_allnames.copy()
df_names.columns = ["name", "images"]
df_names["images"] = pd.to_numeric(df_names["images"], errors="coerce").fillna(0).astype(int)

# peopleDevTrain.csv
df_dev_train = pd.read_csv(CSV_DEV_TRAIN)
df_dev_train.columns = df_dev_train.columns.str.strip()
print(f"peopleDevTrain.csv — {df_dev_train.shape[0]} rows  columns: {list(df_dev_train.columns)}")
print(df_dev_train.head(4).to_string(index=False))
print()

# peopleDevTest.csv
df_dev_test = pd.read_csv(CSV_DEV_TEST)
df_dev_test.columns = df_dev_test.columns.str.strip()
print(f"peopleDevTest.csv  — {df_dev_test.shape[0]} rows  columns: {list(df_dev_test.columns)}")
print(df_dev_test.head(4).to_string(index=False))
print()

print("─" * 50)
print(f"  Total persons       : {len(df_names)}")
print(f"  Total images        : {df_names['images'].sum():,}")
print(f"  Most images         : {df_names['images'].max()}  ({df_names.loc[df_names['images'].idxmax(), 'name']})")
print(f"  Median imgs/person  : {df_names['images'].median():.0f}")
print(f"  Eligible (>={MIN_IMAGES})   : {(df_names['images'] >= MIN_IMAGES).sum()}")
print("─" * 50)

In [ ]:
# ── Distribution plots ────────────────────────────────────────────────────────
eligible_df = df_names[df_names["images"] >= MIN_IMAGES].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# 1. Full distribution (log scale)
axes[0].hist(df_names["images"].values, bins=50, color="steelblue", edgecolor="white")
axes[0].axvline(df_names["images"].median(), color="orange", linestyle="--",
                label=f"Median ({df_names['images'].median():.0f})")
axes[0].axvline(MIN_IMAGES, color="red", linestyle=":", label=f"Min threshold ({MIN_IMAGES})")
axes[0].set_title(f"LFW full distribution ({len(df_names):,} persons)", fontsize=11)
axes[0].set_xlabel("Images per person"); axes[0].set_ylabel("Count")
axes[0].set_yscale("log"); axes[0].legend()

# 2. Eligible persons
axes[1].hist(eligible_df["images"].values, bins=25, color="green", edgecolor="white", alpha=0.8)
axes[1].axvline(MAX_IMAGES, color="red", linestyle="--", label=f"Cap ({MAX_IMAGES})")
axes[1].set_title(f"Eligible persons (>= {MIN_IMAGES} imgs) — {len(eligible_df)} total", fontsize=11)
axes[1].set_xlabel("Images per person"); axes[1].set_ylabel("Count")
axes[1].legend()

# 3. Top 30 (shows skew)
top30 = df_names.nlargest(30, "images")
axes[2].barh(top30["name"].str.replace("_", " ").values[::-1],
             top30["images"].values[::-1], color="steelblue")
axes[2].axvline(MAX_IMAGES, color="red", linestyle="--", label=f"Cap ({MAX_IMAGES})")
axes[2].set_title("Top 30 persons — shows dominance skew", fontsize=11)
axes[2].set_xlabel("Images"); axes[2].legend()

plt.tight_layout()
plt.savefig(WORK_DIR / "dataset_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

## 2. Balanced Class Selection

In [ ]:
# ── Select N_CLASSES with balanced image counts ───────────────────────────────
#
# Strategy:
#   1. Filter persons with >= MIN_IMAGES (from lfw_allnames.csv)
#   2. Sort ascending by image count
#   3. Pick N_CLASSES from the CENTER of the eligible distribution
#      → avoids rarest AND most overrepresented (George-W-Bush problem)
#   4. Verify directory exists on disk, collect actual filenames
#   5. Shuffle + cap at MAX_IMAGES → near-equal class sizes

eligible_df = df_names[df_names["images"] >= MIN_IMAGES].sort_values("images").reset_index(drop=True)

total = len(eligible_df)
center = total // 2
half   = N_CLASSES // 2
start  = max(0, center - half)
end    = start + N_CLASSES
if end > total:
    start = total - N_CLASSES
    end   = total

selected_df = eligible_df.iloc[start:end].reset_index(drop=True)

top20   = []
skipped = []
for _, row in selected_df.iterrows():
    name       = row["name"]
    person_dir = LFW_DIR / name
    if not person_dir.exists():
        skipped.append(name); continue
    imgs = sorted([f.name for f in person_dir.glob("*.jpg")])
    if len(imgs) < MIN_IMAGES:
        skipped.append(name); continue
    random.shuffle(imgs)
    top20.append((name, imgs[:MAX_IMAGES]))

if skipped:
    print(f"WARNING: {len(skipped)} skipped: {skipped}")

assert len(top20) == N_CLASSES, f"Expected {N_CLASSES} classes, got {len(top20)}"

class_names   = [name for name, _ in top20]
class_lengths = [len(imgs) for _, imgs in top20]

print(f"{'─'*65}")
print(f"  {N_CLASSES} classes  |  min={MIN_IMAGES}  cap={MAX_IMAGES}  from lfw_allnames.csv")
print(f"{'─'*65}")
for i, (name, imgs) in enumerate(top20):
    orig = eligible_df.loc[eligible_df["name"] == name, "images"].values[0]
    bar  = "█" * (len(imgs) // 3)
    print(f"  {i:2}. {name:<32} {orig:>4} → {len(imgs):>4}  {bar}")
print(f"{'─'*65}")
print(f"  Total  : {sum(class_lengths)} images")
print(f"  Mean   : {np.mean(class_lengths):.1f}  Std: {np.std(class_lengths):.1f}")
print(f"  CV     : {np.std(class_lengths)/np.mean(class_lengths)*100:.1f}%  (lower = more balanced)")

In [ ]:
# ── Class balance visualization ───────────────────────────────────────────────
names_short = [n[:18] for n, _ in top20]
original    = [eligible_df.loc[eligible_df["name"] == n, "images"].values[0] for n, _ in top20]
capped      = class_lengths

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

x, w = np.arange(N_CLASSES), 0.35
axes[0].bar(x - w/2, original, w, label="Original", color="steelblue", alpha=0.7)
axes[0].bar(x + w/2, capped,   w, label=f"Capped ({MAX_IMAGES})", color="orange")
axes[0].axhline(np.mean(capped), color="red", linestyle="--", label=f"Mean ({np.mean(capped):.0f})")
axes[0].set_xticks(x)
axes[0].set_xticklabels(names_short, rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("Images"); axes[0].set_title("Original vs capped per class")
axes[0].legend()

clr = plt.cm.RdYlGn(np.array(capped[::-1]) / max(capped))
axes[1].barh(names_short[::-1], capped[::-1], color=clr)
axes[1].axvline(np.mean(capped), color="red", linestyle="--", label=f"Mean={np.mean(capped):.0f}")
axes[1].set_xlabel("Images after cap"); axes[1].set_title("Final class distribution")
axes[1].legend()

plt.tight_layout()
plt.savefig(WORK_DIR / "class_balance.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# ── Sample images per class ───────────────────────────────────────────────────
colors = plt.cm.tab20(np.linspace(0, 1, N_CLASSES))
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
for i, (name, imgs) in enumerate(top20):
    img = Image.open(LFW_DIR / name / random.choice(imgs))
    axes.flat[i].imshow(img)
    axes.flat[i].set_title(f"{name.replace('_', ' ')}\n({len(imgs)} imgs)", fontsize=7.5)
    axes.flat[i].axis("off")
    for sp in axes.flat[i].spines.values():
        sp.set_edgecolor(colors[i]); sp.set_linewidth(3); sp.set_visible(True)
plt.suptitle(f"{N_CLASSES} balanced classes — random samples", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(WORK_DIR / "selected_classes.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. Build YOLO Dataset (Train / Val / Test)

In [ ]:
# ── Build YOLO dataset ────────────────────────────────────────────────────────
for split in ["train", "val", "test"]:
    (DATA_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATA_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

stats = {"train": 0, "val": 0, "test": 0}

for class_id, (person_name, img_filenames) in enumerate(top20):
    n       = len(img_filenames)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    splits_map = {
        "train": img_filenames[:n_train],
        "val":   img_filenames[n_train:n_train + n_val],
        "test":  img_filenames[n_train + n_val:],
    }
    for split, files in splits_map.items():
        for fname in files:
            stem = f"{person_name}_{Path(fname).stem}"
            shutil.copy2(LFW_DIR / person_name / fname,
                         DATA_DIR / "images" / split / f"{stem}.jpg")
            # YOLO annotation: full-image bbox (LFW = face-centered images)
            (DATA_DIR / "labels" / split / f"{stem}.txt").write_text(
                f"{class_id} 0.5 0.5 1.0 1.0\n")
            stats[split] += 1

data_yaml = {
    "path":  str(DATA_DIR),
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc":    N_CLASSES,
    "names": class_names,
}
yaml_path = DATA_DIR / "data.yaml"
yaml_path.write_text(yaml.dump(data_yaml, default_flow_style=False, allow_unicode=True))

total = sum(stats.values())
print(f"Dataset ready — {N_CLASSES} classes")
for split, n in stats.items():
    print(f"  {split:<6}: {n:>4} images  ({n/total*100:.1f}%)")
print(f"  Total : {total}")

In [ ]:
# ── Per-class split distribution ──────────────────────────────────────────────
split_counts = {}
for split in ["train", "val", "test"]:
    cc = {c: 0 for c in range(N_CLASSES)}
    for f in (DATA_DIR / "labels" / split).glob("*.txt"):
        cc[int(f.read_text().split()[0])] += 1
    split_counts[split] = [cc[c] for c in range(N_CLASSES)]

x, w = np.arange(N_CLASSES), 0.28
fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(x - w, split_counts["train"], w, label="Train",  color="steelblue")
ax.bar(x,     split_counts["val"],   w, label="Val",    color="orange")
ax.bar(x + w, split_counts["test"],  w, label="Test",   color="green")
ax.set_xticks(x)
ax.set_xticklabels([n[:14] for n in class_names], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Images"); ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_title("Train / Val / Test distribution per class")
plt.tight_layout()
plt.savefig(WORK_DIR / "split_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. YOLO26m Training

In [ ]:
# ── Model info ────────────────────────────────────────────────────────────────
model = YOLO(BASE_MODEL)
print(model.info())

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,

    # Augmentation (face-targeted)
    flipud=0.0,        # never flip vertically (faces)
    fliplr=0.5,        # horizontal mirror OK
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=15.0,      # rotation ±15°
    translate=0.1,
    scale=0.4,
    shear=2.0,
    perspective=0.0002,
    mosaic=0.5,
    mixup=0.1,
    copy_paste=0.0,
    erasing=0.3,       # random erase for occlusion robustness

    # Optimizer
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.005,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=5,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,

    # Regularization
    dropout=0.1,
    label_smoothing=0.1,

    # Save & stop
    patience=20,
    save_period=10,
    project=str(MODEL_DIR),
    name=RUN_NAME,
    plots=True,
    verbose=True,
    seed=SEED,
    deterministic=True,
)

In [ ]:
# ── Copy best weights ─────────────────────────────────────────────────────────
best_pt = MODEL_DIR / RUN_NAME / "weights" / "best.pt"
dest    = MODEL_DIR / "face_yolo.pt"

if best_pt.exists():
    shutil.copy2(best_pt, dest)
    # Also copy to /kaggle/working/ root so kaggle CLI can download it
    shutil.copy2(best_pt, WORK_DIR / "face_yolo.pt")
    print(f"✓ Best model saved: {dest}  ({dest.stat().st_size/1e6:.1f} MB)")
    print(f"✓ Also available at: {WORK_DIR / 'face_yolo.pt'}")
else:
    print("✗ best.pt not found — check training logs")

## 5. Training Curves

In [ ]:
# ── Training curves from results.csv ─────────────────────────────────────────
results_csv = MODEL_DIR / RUN_NAME / "results.csv"

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    plots = [
        ("train/box_loss", "val/box_loss",          "Box Loss"),
        ("train/cls_loss", "val/cls_loss",          "Classification Loss"),
        ("train/dfl_loss", "val/dfl_loss",          "DFL Loss"),
        ("metrics/precision(B)", None,              "Precision"),
        ("metrics/recall(B)",    None,              "Recall"),
        ("metrics/mAP50(B)",     "metrics/mAP50-95(B)", "mAP"),
    ]
    for ax, (col_a, col_b, title) in zip(axes.flat, plots):
        if col_a in df.columns:
            ax.plot(df["epoch"], df[col_a], label="Train", color="steelblue")
        if col_b and col_b in df.columns:
            ax.plot(df["epoch"], df[col_b], label="mAP50-95" if "mAP" in title else "Val",
                    color="orange", linestyle="--")
        ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)

    plt.suptitle("Training Curves — YOLO26m Face Recognition", fontsize=13)
    plt.tight_layout()
    plt.savefig(WORK_DIR / "training_curves.png", dpi=120, bbox_inches="tight")
    plt.show()

    best_ep = df["metrics/mAP50(B)"].idxmax()
    print(f"Best epoch : {int(df.loc[best_ep, 'epoch'])}")
    for col in ["metrics/mAP50(B)", "metrics/mAP50-95(B)", "metrics/precision(B)", "metrics/recall(B)"]:
        if col in df.columns:
            print(f"  {col.split('/')[-1]:<20} {df.loc[best_ep, col]:.4f}")
else:
    png = MODEL_DIR / RUN_NAME / "results.png"
    if png.exists(): display(IPImage(str(png)))

## 6. Test Set Evaluation

In [ ]:
# ── Metrics on the independent test set ──────────────────────────────────────
best_model = YOLO(str(dest))

test_yaml = data_yaml.copy()
test_yaml["val"] = "images/test"
test_yaml_path = DATA_DIR / "data_test.yaml"
test_yaml_path.write_text(yaml.dump(test_yaml, default_flow_style=False, allow_unicode=True))

test_metrics = best_model.val(
    data=str(test_yaml_path), split="val",
    verbose=True, plots=True, save_json=True,
)

print("\n" + "═"*45)
print("  TEST SET RESULTS")
print("═"*45)
print(f"  mAP50     : {test_metrics.box.map50:.4f}")
print(f"  mAP50-95  : {test_metrics.box.map:.4f}")
print(f"  Precision : {test_metrics.box.mp:.4f}")
print(f"  Recall    : {test_metrics.box.mr:.4f}")
print("═"*45)
print("\n  mAP50 per class:")
for name, ap in zip(class_names, test_metrics.box.ap50):
    print(f"  {name[:28]:<28}  {'█'*int(ap*30):<30}  {ap:.3f}")

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
cm_png = list((MODEL_DIR / RUN_NAME).glob("**/confusion_matrix_normalized.png"))
if cm_png:
    display(IPImage(str(cm_png[0])))
else:
    from sklearn.metrics import confusion_matrix, classification_report
    y_true, y_pred = [], []
    for lbl in (DATA_DIR / "labels" / "test").glob("*.txt"):
        img_f = DATA_DIR / "images" / "test" / lbl.with_suffix(".jpg").name
        if not img_f.exists(): continue
        y_true.append(int(lbl.read_text().split()[0]))
        pr = best_model.predict(str(img_f), verbose=False, conf=0.3)[0]
        y_pred.append(int(pr.boxes.cls[0]) if pr.boxes and len(pr.boxes) else N_CLASSES)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(N_CLASSES)))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                xticklabels=[n[:10] for n in class_names],
                yticklabels=[n[:10] for n in class_names])
    plt.title("Normalized Confusion Matrix — YOLO26m (test set)", fontsize=13)
    plt.ylabel("True class"); plt.xlabel("Predicted class")
    plt.tight_layout()
    plt.savefig(WORK_DIR / "confusion_matrix.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

## 7. Inference — Test Set Predictions

In [ ]:
# ── 20 random test predictions ────────────────────────────────────────────────
test_imgs   = list((DATA_DIR / "images" / "test").glob("*.jpg"))
sample_imgs = random.sample(test_imgs, min(20, len(test_imgs)))

fig, axes = plt.subplots(4, 5, figsize=(18, 14))
correct = total_inf = 0

for i, img_path in enumerate(sample_imgs):
    lbl_path   = DATA_DIR / "labels" / "test" / img_path.with_suffix(".txt").name
    true_class = int(lbl_path.read_text().split()[0]) if lbl_path.exists() else -1
    true_name  = class_names[true_class] if true_class >= 0 else "?"
    pred       = best_model.predict(str(img_path), verbose=False, conf=0.25)[0]
    img_np     = np.array(Image.open(img_path))

    if pred.boxes and len(pred.boxes):
        pred_class = int(pred.boxes.cls[0])
        pred_conf  = float(pred.boxes.conf[0])
        pred_name  = class_names[pred_class]
        ok = pred_class == true_class
    else:
        pred_name, pred_conf, ok = "Unknown", 0.0, False

    correct += int(ok); total_inf += 1
    color  = "#00cc44" if ok else "#cc0000"
    axes.flat[i].imshow(img_np); axes.flat[i].axis("off")
    axes.flat[i].set_title(f"{'✓' if ok else '✗'} {pred_name[:14]}\n({pred_conf:.0%})  gt:{true_name[:12]}",
                            fontsize=7.5, color=color)
    for sp in axes.flat[i].spines.values():
        sp.set_edgecolor(color); sp.set_linewidth(3); sp.set_visible(True)

acc = correct / total_inf * 100
plt.suptitle(f"Test set inference — sample accuracy: {acc:.1f}% ({correct}/{total_inf})",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(WORK_DIR / "test_predictions.png", dpi=120, bbox_inches="tight")
plt.show()

## 8. Model Export

In [ ]:
# ── Export to ONNX ────────────────────────────────────────────────────────────
onnx_path = best_model.export(format="onnx", imgsz=IMGSZ, simplify=True)
shutil.copy2(onnx_path, WORK_DIR / "face_yolo.onnx")
print(f"ONNX: {WORK_DIR / 'face_yolo.onnx'  }")

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print("═"*55)
print("  FINAL SUMMARY")
print("═"*55)
print(f"  Model           : YOLO26m (Jan. 2026)")
print(f"  Classes         : {N_CLASSES} (LFW balanced)")
print(f"  Train images    : {stats['train']}")
print(f"  Val images      : {stats['val']}")
print(f"  Test images     : {stats['test']}")
print(f"  mAP50  (test)   : {test_metrics.box.map50:.4f}")
print(f"  mAP50-95 (test) : {test_metrics.box.map:.4f}")
print(f"  Precision       : {test_metrics.box.mp:.4f}")
print(f"  Recall          : {test_metrics.box.mr:.4f}")
print()
print("  Output files:")
for f in sorted(WORK_DIR.glob("*.pt")) + sorted(WORK_DIR.glob("*.onnx")) + sorted(WORK_DIR.glob("*.png")):
    print(f"    {f.name:<42} {f.stat().st_size/1e6:6.1f} MB")
print("═"*55)
print()
print("  To use the model:")
print("  1. Download face_yolo.pt from the Output tab")
print("  2. Place in models/face_yolo.pt")
print("  3. streamlit run streamlit_app.py")